# 第 1 章 · `MemoryStore` 双态（可执行源码副本）

对照：`../hermes_src/tools/memory_tool.py` → `class MemoryStore`

**真实数据**：`fixtures/MEMORY.md` + `fixtures/USER.md`（从本机 `~/.hermes/memories/` 拷贝）

**讲解要点**：Session 启动 `load_from_disk` 冻 `_system_prompt_snapshot`；同 session 的 `add()` 只改 live，**不改** snapshot → Prompt Cache 前缀稳定。

```mermaid
sequenceDiagram
    participant Disk as MEMORY.md / USER.md
    participant Boot as load_from_disk
    participant Store as MemoryStore
    participant SP as format_for_system_prompt
    Disk->>Boot: 按 § 拆条目
    Boot->>Store: 写入 _system_prompt_snapshot
    SP->>Store: 只读 snapshot
    Note over Store: add() 改 live entries
    Note over SP: snapshot 仍是启动时那份
```


## ① 可执行源码副本（先 Run 这一格）

下面从 Hermes `MemoryStore` **拷贝核心逻辑**，去掉磁盘锁 / threat scan / drift，保留你要讲的实现不变量。

In [5]:
# =============================================================================
# SOURCE COPY: hermes_src/tools/memory_tool.py :: MemoryStore
# 保留：双态、load_from_disk 冻 snapshot、add 只改 live、format_for_system_prompt 读 snapshot
# 省略：fcntl 锁、threat_patterns、外部 drift 检测（讲主逻辑不需要）
# =============================================================================
from __future__ import annotations

from pathlib import Path
from typing import Any, Dict, List, Optional

# 真源码常量：条目之间用 "\n§\n" 分隔（不是空行）
ENTRY_DELIMITER = "\n§\n"

# 本 notebook 旁的真实 MEMORY.md / USER.md 副本（从 ~/.hermes/memories/ 拷来）
FIXTURES = Path("fixtures")


class MemoryStore:
    """
    Bounded curated memory. One instance per AIAgent.

    Maintains two parallel states:
      - _system_prompt_snapshot: frozen at load time, used for system prompt injection.
        Never mutated mid-session. Keeps prefix cache stable.
      - memory_entries / user_entries: live state, mutated by tool calls.
        Tool responses always reflect this live state.
    """

    _MAX_CONSOLIDATION_FAILURES_PER_TURN = 3

    def __init__(self, memory_char_limit: int = 2200, user_char_limit: int = 1375):
        # ---- live 态：本 session 内可被 memory tool 读写 ----
        self.memory_entries: List[str] = []   # 对应 MEMORY.md 条目
        self.user_entries: List[str] = []     # 对应 USER.md 条目
        self.memory_char_limit = memory_char_limit
        self.user_char_limit = user_char_limit
        # ---- frozen 态：只在 load() 时写一次，供 system prompt 注入 ----
        # 不变量：同 session 内 add/replace/remove 都不碰这份 dict → Prompt Cache 前缀稳定
        self._system_prompt_snapshot: Dict[str, str] = {"memory": "", "user": ""}
        self._consolidation_failures = 0

    # -- load_from_disk()：读 MEMORY.md / USER.md，然后冻结 snapshot ------------

    def load_from_disk(self, mem_dir: Path | None = None):
        """从磁盘读真实 MEMORY.md / USER.md，灌入 live，并冻结 system prompt snapshot。

        对应真源码 MemoryStore.load_from_disk()（省略 threat sanitize）。
        """
        mem_dir = Path(mem_dir) if mem_dir else FIXTURES
        self.memory_entries = self._read_file(mem_dir / "MEMORY.md")
        self.user_entries = self._read_file(mem_dir / "USER.md")
        # dict.fromkeys：去重且保序（与真源码一致）
        self.memory_entries = list(dict.fromkeys(self.memory_entries))
        self.user_entries = list(dict.fromkeys(self.user_entries))
        # ★ 关键点：把 live 渲染成字符串，冻进 snapshot —— 之后本 session 不再更新
        self._system_prompt_snapshot = {
            "memory": self._render_block("memory", self.memory_entries),
            "user": self._render_block("user", self.user_entries),
        }

    @staticmethod
    def _read_file(path: Path) -> List[str]:
        """按 ENTRY_DELIMITER（\\n§\\n）拆成条目列表；文件不存在或空 → []。"""
        if not path.exists():
            return []
        try:
            raw = path.read_text(encoding="utf-8")
        except (OSError, IOError):
            return []
        if not raw.strip():
            return []
        # 必须用完整 delimiter 拆，不能只按 "§"（条目内容里也可能有 §）
        entries = [e.strip() for e in raw.split(ENTRY_DELIMITER)]
        return [e for e in entries if e]

    # -- 对应 add()：只改 live +（真源码还会 save_to_disk）；绝不改 snapshot -----

    def add(self, target: str, content: str) -> Dict[str, Any]:
        """Append a new entry. Returns error if it would exceed the char limit."""
        content = content.strip()
        if not content:
            return {"success": False, "error": "Content cannot be empty."}

        entries = self._entries_for(target)   # live 列表（user_entries 或 memory_entries）
        limit = self._char_limit(target)

        # 去重：已存在则当作成功，不重复追加
        if content in entries:
            return self._success_response(target, "Entry already exists (no duplicate added).")

        # 容量预检：用 ENTRY_DELIMITER 拼完再量长度（与落盘格式一致）
        new_entries = entries + [content]
        new_total = len(ENTRY_DELIMITER.join(new_entries))
        if new_total > limit:
            current = self._char_count(target)
            return {
                "success": False,
                "error": (
                    f"Memory at {current:,}/{limit:,} chars. "
                    f"Adding this entry ({len(content)} chars) would exceed the limit."
                ),
                "current_entries": list(entries),
            }

        entries.append(content)
        self._set_entries(target, entries)  # 只改 live
        # NOTE: 真源码此处 save_to_disk(target) —— 落盘供下一 session load
        # NOTE: 故意不更新 _system_prompt_snapshot —— 这就是 Prompt Cache 不变量
        return self._success_response(target, "Entry added.")

    # -- 对应 format_for_system_prompt()：永远读冻结快照 ----------------------

    def format_for_system_prompt(self, target: str) -> Optional[str]:
        """
        Return the frozen snapshot for system prompt injection.

        This returns the state captured at load time, NOT the live state.
        Mid-session writes do not affect this. This keeps the system prompt
        stable across all turns, preserving the prefix cache.
        """
        # ★ 读 snapshot，不读 memory_entries / user_entries
        block = self._system_prompt_snapshot.get(target, "")
        return block if block else None

    def system_prompt_memory_section(self) -> str:
        """拼 MEMORY + USER 两块冻结文本，模拟注入 system prompt 的完整 memory 段。"""
        parts = [p for t in ("memory", "user") if (p := self.format_for_system_prompt(t))]
        return "\n\n".join(parts)

    # -- helpers（对齐真源码命名）----------------------------------------------

    def _entries_for(self, target: str) -> List[str]:
        """按 target 返回对应 live 列表的引用（可原地 append）。"""
        return self.user_entries if target == "user" else self.memory_entries

    def _set_entries(self, target: str, entries: List[str]) -> None:
        """写回 live；不碰 _system_prompt_snapshot。"""
        if target == "user":
            self.user_entries = entries
        else:
            self.memory_entries = entries

    def _char_limit(self, target: str) -> int:
        """USER.md / MEMORY.md 各自有独立字符上限。"""
        return self.user_char_limit if target == "user" else self.memory_char_limit

    def _char_count(self, target: str) -> int:
        """当前 live 占用：条目用 ENTRY_DELIMITER 拼接后的长度。"""
        return len(ENTRY_DELIMITER.join(self._entries_for(target)))

    def _success_response(self, target: str, message: str = None) -> Dict[str, Any]:
        """工具成功回包；note 明确提醒：snapshot 本 session 未变。"""
        self._consolidation_failures = 0
        entries = self._entries_for(target)
        current = self._char_count(target)
        limit = self._char_limit(target)
        pct = min(100, int((current / limit) * 100)) if limit > 0 else 0
        resp = {
            "success": True,
            "done": True,
            "target": target,
            "usage": f"{pct}% — {current:,}/{limit:,} chars",
            "entry_count": len(entries),
            # 给模型/讲解看的提示：写成功 ≠ system prompt 立刻更新
            "note": "Write saved. Snapshot for system prompt is UNCHANGED this session.",
        }
        if message:
            resp["message"] = message
        return resp

    def _render_block(self, target: str, entries: List[str]) -> str:
        """把条目列表渲染成带 header / 用量的 system prompt 块（load 时调用一次）。"""
        if not entries:
            return ""
        limit = self._char_limit(target)
        content = ENTRY_DELIMITER.join(entries)
        current = len(content)
        pct = min(100, int((current / limit) * 100)) if limit > 0 else 0
        if target == "user":
            header = f"USER PROFILE (who the user is) [{pct}% — {current:,}/{limit:,} chars]"
        else:
            header = f"MEMORY (your personal notes) [{pct}% — {current:,}/{limit:,} chars]"
        separator = "═" * 46
        return f"{separator}\n{header}\n{separator}\n{content}"



print("MemoryStore 已定义（源码副本可执行）")

MemoryStore 已定义（源码副本可执行）


## ② 演示：讲「冻 snapshot → add 不改 SP」

In [6]:
import hashlib

store = MemoryStore()

# Session 启动：从真实 MEMORY.md / USER.md 读盘并冻结 snapshot
# （对应真源码 load_from_disk()；fixtures/ 是你本机 ~/.hermes/memories/ 的副本）
print("读取:", (FIXTURES / "MEMORY.md").resolve())
print("读取:", (FIXTURES / "USER.md").resolve())
store.load_from_disk(FIXTURES)

print(f"\nlive memory_entries ({len(store.memory_entries)} 条):")
for i, e in enumerate(store.memory_entries):
    print(f"  [{i}] {e[:80]}{'...' if len(e) > 80 else ''}")
print(f"\nlive user_entries ({len(store.user_entries)} 条):")
for i, e in enumerate(store.user_entries):
    print(f"  [{i}] {e[:80]}{'...' if len(e) > 80 else ''}")

# 取出将注入 system prompt 的冻结文本，并算指纹，便于对比后续 add 后是否仍不变
sp0 = store.system_prompt_memory_section()
fp0 = hashlib.sha256(sp0.encode()).hexdigest()[:16]
print("\n=== Session 启动注入 SP 的 snapshot ===")
print(sp0)
print("\nfingerprint =", fp0)
# 下一格会 add 新条目，再比一次 fingerprint —— 期望仍等于 fp0


读取: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\08-hermes-agent\04.1-memory\notebooks\fixtures\MEMORY.md
读取: D:\workspace\doc\面试狂魔\人工智能面试题\AI_coding_interview\08-hermes-agent\04.1-memory\notebooks\fixtures\USER.md

live memory_entries (2 条):
  [0] Julie 偏好严谨导师型面试辅导，标准回答结构：面试官考察意图 → 标准回答 → 深层解析 → 加分点 → 关联她的项目经验。喜欢分场景整理（写了 TUV_二...
  [1] Julie 面了中通 AI Product Manager 挂了，发现自己不适合 PM 方向，更喜欢也擅长开发。应聚焦 AI Agent Engineer / ...

live user_entries (2 条):
  [0] # Julie - 用户档案

## 基本信息
- 称呼：Julie
- 角色：软件工程师，正在寻找 AI Agent Engineer 岗位
- 编程语言：P...
  [1] Julie，软件工程师，正在找 AI Agent Engineer 工作。Python 主力 + C 背景。持有 CCIE 安全认证。做过 RAG Agent ...

=== Session 启动注入 SP 的 snapshot ===
══════════════════════════════════════════════
MEMORY (your personal notes) [11% — 259/2,200 chars]
══════════════════════════════════════════════
Julie 偏好严谨导师型面试辅导，标准回答结构：面试官考察意图 → 标准回答 → 深层解析 → 加分点 → 关联她的项目经验。喜欢分场景整理（写了 TUV_二面场景准备.md）。面试遇到不会的问题会带回来深挖，需要面试话术级别的回答。
§
Julie 面了中通 AI Product Manager 挂了，发现自己不适合 PM 方向，更喜欢也擅长开发。应聚焦 AI

In [7]:
# 同 session：memory tool add —— live 变了，format_for_system_prompt 不变
# （模拟 agent 调用 memory(action=add, target=memory, ...)）
NEW = "本 notebook 演示：同 session add 不改 system prompt snapshot"
result = store.add("memory", NEW)
print("add result:", result)

# live 已包含新条目（工具回包 / 后续 tool 读到的都是这份）
print("\nlive memory_entries 条数 =", len(store.memory_entries))
print("最后一条 =", store.memory_entries[-1])

# 但注入 system prompt 的仍是 load_from_disk 时冻的那份 —— 没有 NEW
print("\nformat_for_system_prompt('memory') 仍是启动快照（不含刚 add 的条目）:")
print(store.format_for_system_prompt("memory")[:400], "...")

# 用指纹证明：整段 SP memory section 字节级不变 → Prompt Cache 前缀可命中
sp1 = store.system_prompt_memory_section()
fp1 = hashlib.sha256(sp1.encode()).hexdigest()[:16]
print("\nfingerprint 不变?", fp0 == fp1, fp0)
# 结论：新内容要等下一 session 的 load_from_disk() 才会冻进 snapshot 并进入 system prompt


add result: {'success': True, 'done': True, 'target': 'memory', 'usage': '14% — 315/2,200 chars', 'entry_count': 3, 'note': 'Write saved. Snapshot for system prompt is UNCHANGED this session.', 'message': 'Entry added.'}

live memory_entries 条数 = 3
最后一条 = 本 notebook 演示：同 session add 不改 system prompt snapshot

format_for_system_prompt('memory') 仍是启动快照（不含刚 add 的条目）:
══════════════════════════════════════════════
MEMORY (your personal notes) [11% — 259/2,200 chars]
══════════════════════════════════════════════
Julie 偏好严谨导师型面试辅导，标准回答结构：面试官考察意图 → 标准回答 → 深层解析 → 加分点 → 关联她的项目经验。喜欢分场景整理（写了 TUV_二面场景准备.md）。面试遇到不会的问题会带回来深挖，需要面试话术级别的回答。
§
Julie 面了中通 AI Product Manager 挂了，发现自己不适合 PM 方向，更喜欢也擅长开发。应聚焦 AI Agent Engineer / RAG Engineer 等纯开发岗，不投 PM。目前 TUV 仪器仪表 AI 工程师已过一面，在准备二 ...

fingerprint 不变? True 42a4da802c461b6a


In [8]:
print(store.format_for_system_prompt("memory"))

══════════════════════════════════════════════
MEMORY (your personal notes) [11% — 259/2,200 chars]
══════════════════════════════════════════════
Julie 偏好严谨导师型面试辅导，标准回答结构：面试官考察意图 → 标准回答 → 深层解析 → 加分点 → 关联她的项目经验。喜欢分场景整理（写了 TUV_二面场景准备.md）。面试遇到不会的问题会带回来深挖，需要面试话术级别的回答。
§
Julie 面了中通 AI Product Manager 挂了，发现自己不适合 PM 方向，更喜欢也擅长开发。应聚焦 AI Agent Engineer / RAG Engineer 等纯开发岗，不投 PM。目前 TUV 仪器仪表 AI 工程师已过一面，在准备二面（线下）。


**口述稿**：`load_from_disk` 从真实 `MEMORY.md` / `USER.md`（`§` 分隔）读入并冻 snapshot；`add` 成功只证明 live 更新了；`format_for_system_prompt` 仍读 `_system_prompt_snapshot`，所以当前 session 的 system 前缀字节不变，缓存可命中。下一 session 再 `load_from_disk` 才会把新内容冻进 snapshot。
